# quant-kit — GGUF Quantization Pipeline

> **GitHub**: [DhruvalPtl/quant-kit](https://github.com/DhruvalPtl/quant-kit)  
> **HuggingFace**: [Dhptl](https://huggingface.co/Dhptl)

Run each cell **top to bottom**. If a cell fails, paste the error output in our chat.

---
### Before you start:
1. **Runtime → Change runtime type → GPU (T4)**
2. **Add your HF token to Colab Secrets** (key icon in left sidebar):
   - Name: `HF_TOKEN`  
   - Value: your token from https://huggingface.co/settings/tokens


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 1: Check runtime (GPU / disk / RAM)
# ─────────────────────────────────────────────────────────────
import subprocess, shutil, psutil

# GPU check
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'[OK] GPU: {gpu.stdout.strip()}')
else:
    print('[!] No GPU detected!')
    print('    Go to Runtime > Change runtime type > GPU')

# Disk check
disk = shutil.disk_usage('/')
print(f'[OK] Disk: {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB')

# RAM check
ram = psutil.virtual_memory()
print(f'[OK] RAM:  {ram.available/1e9:.1f} GB available of {ram.total/1e9:.1f} GB')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 2: Clone quant-kit & run setup
# ─────────────────────────────────────────────────────────────
import os

REPO = 'https://github.com/DhruvalPtl/quant-kit.git'
WORKDIR = '/content/quant-kit'

if os.path.exists(WORKDIR):
    print('[OK] quant-kit already cloned — pulling latest...')
    os.system(f'git -C {WORKDIR} pull')
else:
    print('[->] Cloning quant-kit...')
    os.system(f'git clone {REPO} {WORKDIR}')

os.chdir(WORKDIR)
print(f'[OK] Working directory: {os.getcwd()}')

# Run Linux setup (downloads llama.cpp binaries + conversion scripts)
!python setup_linux.py

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 3: HuggingFace authentication
# ─────────────────────────────────────────────────────────────
import os
from google.colab import userdata

# Load token from Colab Secrets (never hardcode your token!)
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token

    # Write to .env so config.py picks it up
    with open('/content/quant-kit/.env', 'w') as f:
        f.write(f'hf_token = "{hf_token}"\n')

    # Verify token works
    from huggingface_hub import HfApi
    user = HfApi(token=hf_token).whoami()
    print(f'[OK] Logged in as: {user["name"]}')

except Exception as e:
    print(f'[ERR] Token error: {e}')
    print('  1. Click the key icon in the left sidebar')
    print('  2. Add secret: Name=HF_TOKEN, Value=your_token')
    print('  3. Enable "Notebook access" toggle')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 4: Quantize
# Edit MODEL_ID below before running!
# ─────────────────────────────────────────────────────────────

MODEL_ID = 'google/gemma-4-12B'   # <-- change this for different models
QUANTS   = 'Q4_K_M Q5_K_M Q8_0 IQ4_XS'  # <-- quant types to produce

# --delete-src frees ~24GB after FP16 conversion (important on Colab!)
!python quantize.py \
    --model {MODEL_ID} \
    --quants {QUANTS} \
    --delete-src

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 5: Benchmark
# ─────────────────────────────────────────────────────────────

# Model folder name = last part of MODEL_ID
MODEL_NAME = MODEL_ID.split('/')[-1]

# ngl=99 offloads all layers to GPU (T4 has 15GB VRAM — enough for 12B Q4)
!python benchmark.py --model {MODEL_NAME} --ngl 99

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 6: Generate model card
# ─────────────────────────────────────────────────────────────

HF_AUTHOR = 'Dhptl'   # <-- your HuggingFace username

!python model_card.py \
    --model {MODEL_NAME} \
    --original {MODEL_ID} \
    --author {HF_AUTHOR}

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 7: Upload to HuggingFace
# Creates repo Dhptl/gemma-4-12B-GGUF automatically
# ─────────────────────────────────────────────────────────────

!python upload.py \
    --model {MODEL_NAME} \
    --author {HF_AUTHOR}

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 8: (Optional) Disk cleanup
# Run this after upload to free disk for the next model
# ─────────────────────────────────────────────────────────────
import shutil, os

model_output = f'/content/quant-kit/output/{MODEL_NAME}'
if os.path.exists(model_output):
    shutil.rmtree(model_output)
    print(f'[OK] Cleaned up: {model_output}')

# Check remaining disk
disk = shutil.disk_usage('/')
print(f'[OK] Disk after cleanup: {disk.free/1e9:.1f} GB free')
print()
print('Ready for next model! Update MODEL_ID in Cell 4 and re-run from Cell 4.')